In [1]:
!mkdir data/fsc22

In [12]:
ROOT = "../DATA/FSC22"
AUDIO = ROOT + "/Audio"
META = ROOT + "/Metadata"

In [ ]:
from utils.io import list_audio

AUDIO_EXTS = (".wav", ".flac", ".ogg", ".mp3", ".m4a")

audio_paths = list_audio(AUDIO)

Walking through: ../DATA/FSC22/Audio
Found 2025 .wav files
Found 0 .flac files
Found 0 .ogg files
Found 0 .mp3 files
Found 0 .m4a files


### Prepare images

- Resampled → 44.1 kHz
- Duration → 5 s
- Padding → ZERO
- Truncation → NONE

#### Augmentation

- Waveform-level → stochastic perturbations (additive Gaussian noise, time-stretching, pitch shifting, random gain)
- Transform into 2D spectrogram (Short-Time Fourier Transform, 1024-window, 256 hop)
- Map to 40 Mel Freq. bins, converted to log decibel scale
- Normalized using mean and st. deviation from training dataset.
- Spectrogram-level → time&frequency masking, on-the-fly Mixup augmentation (synthesize interpolation training pairs and regularize)

### Optimization

– AdamW 32 → batch
– cross entropy loss,

- lr = 1e-3
- weight decay → 1e-4
- Cosine annealing scheduler
- Early stopping → 100 patience

---

Run:

```bash
cd soundedge-esc-application
python -m training.train \
 --csv ../../data/fsc22/5-fold.csv \
 --audio-dir /YOUR/fsc22/audio \
 --val-fold 5 --epochs 300 --out weights/fsc22_model.pth
```


In [1]:
!python soundedge-esc-application/training/train.py \
    --csv ./data/fsc22/5-fold.csv \
    --audio-dir ../DATA/FSC22/Audio/ \
    --val-fold 5 --epochs 300 --out weights/fsc22_model.pth \
    --batch-size 32 \
    --accum-steps 1 \
    --num-workers 8

15:47:33 INFO [main] args: {'csv': './data/fsc22/5-fold.csv', 'audio_dir': '../DATA/FSC22/Audio/', 'val_fold': 5, 'epochs': 300, 'batch_size': 32, 'accum_steps': 1, 'lr': 0.001, 'weight_decay': 0.0001, 'mixup_alpha': 0.2, 'patience': 100, 'num_workers': 8, 'out': 'weights/fsc22_model.pth', 'stats': 'stats/fsc22_mel_stats.json', 'recompute_stats': False, 'device': 'cuda'}
15:47:33 INFO [main] STAGE: build label map
15:47:33 INFO [main] classes=26 train_folds=[1, 2, 3, 4] val_fold=5
15:47:33 INFO [main] STAGE: normalization stats
15:47:33 INFO [main] reusing cached stats: stats/fsc22_mel_stats.json
15:47:33 INFO [main] stats mean=-16.7370 std=22.7698
15:47:33 INFO [main] STAGE: build datasets / loaders
15:47:33 INFO [main] train_ds=1560 val_ds=390 batches/epoch≈48
15:47:33 INFO [main] STAGE: build model (device=cuda)
15:47:40 INFO [run_epoch] [train] first batch x=(32, 1, 40, 862) y=(32,)
15:47:40 INFO [run_epoch] [train] peak RSS: 742 MB
15:47:40 INFO [run_epoch] [train] VRAM allocated:

In [ ]:
!cd soundedge-esc-application && \
    python -m compression.optimize --smoke

13:30:14 INFO [main] device=cuda  smoke=True
13:30:14 INFO [main] num_classes=27  test_fold=None
13:30:14 WARNING [main] teacher weights not loaded (smoke or missing): weights/fsc22_model.pth
13:30:15 INFO [main] TEACHER  params=512191  val_acc=0.000
13:30:15 INFO [main] PRUNED   params=169215  (one-shot amount=0.50, fc reset -> needs KD)
13:30:16 INFO [train_distill] [KD] epoch 1/1  loss=1.7482  val_acc=0.062
13:30:16 INFO [train_distill] [KD] best val_acc=0.062
13:30:16 INFO [main] KD pruned fp32 saved -> weights/fsc22_pruned_distilled.pth (0.69 MB)
forest-sounds/.venv/lib/python3.11/site-packages/torch/ao/quantization/observer.py:534: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super().__init__(
13:30:16 INFO [train_distill] [QAT] epoch 1/1  loss=1.1946  val_acc=0.125
13:30:16 INFO [train_distill] [QAT] best val_acc=0.125
13:30:16 WARNING [main] torchscript e

In [3]:
!cd soundedge-esc-application && \
    python -m compression.optimize --smoke --paper-mode

13:31:11 INFO [main] device=cuda  smoke=True
13:31:11 INFO [main] num_classes=27  test_fold=None
13:31:11 WARNING [main] teacher weights not loaded (smoke or missing): weights/fsc22_model.pth
13:31:12 INFO [main] TEACHER  params=512191  val_acc=0.000
13:31:12 INFO [main] PAPER-MODE: iterative asymmetric prune (step=0.10, rounds<=8, target=off, rewind=2ep)
13:31:12 INFO [main]   round 1: params=468430
13:31:13 INFO [train_distill] [rewind1] epoch 1/2  loss=1.8016  val_acc=0.062
13:31:13 INFO [train_distill] [rewind1] epoch 2/2  loss=1.3252  val_acc=0.062
13:31:13 INFO [train_distill] [rewind1] best val_acc=0.062
13:31:13 INFO [main]   round 2: params=428848
13:31:14 INFO [train_distill] [rewind2] epoch 1/2  loss=1.5183  val_acc=0.125
13:31:14 INFO [train_distill] [rewind2] epoch 2/2  loss=1.3276  val_acc=0.062
13:31:14 INFO [train_distill] [rewind2] best val_acc=0.125
13:31:14 INFO [main]   round 3: params=396853
13:31:14 INFO [train_distill] [rewind3] epoch 1/2  loss=1.8088  val_acc=0.

### Optimization

In [6]:
!cd soundedge-esc-application && ls ../../DATA/

FSC22


In [7]:
!cd soundedge-esc-application && \
    python -m compression.optimize \
        --csv ../data/fsc22/5-fold.csv \
        --audio-dir ../../DATA/FSC22/Audio/ \
        --weights weights/fsc22_model.pth \
        --out-dir  weights/optimized \
        --val-fold 5 \
        --prune-amount 0.5 \
        --kd-epochs 50 \
        --qat-epochs 15 \
        --batch-size 32 --num-workers 8

13:36:57 INFO [main] device=cuda  smoke=False
13:36:57 INFO [main] num_classes=26  test_fold=None
13:36:57 WARNING [main] teacher weights not loaded (smoke or missing): weights/fsc22_model.pth
13:36:58 INFO [main] TEACHER  params=511805  val_acc=0.015
13:36:58 INFO [main] PRUNED   params=168829  (one-shot amount=0.50, fc reset -> needs KD)
13:37:24 INFO [train_distill] [KD] epoch 1/50  loss=1.3942  val_acc=0.115
13:37:47 INFO [train_distill] [KD] epoch 2/50  loss=1.2670  val_acc=0.182
13:38:09 INFO [train_distill] [KD] epoch 3/50  loss=1.2417  val_acc=0.241
13:38:34 INFO [train_distill] [KD] epoch 4/50  loss=1.2224  val_acc=0.290
13:38:54 INFO [train_distill] [KD] epoch 5/50  loss=1.2020  val_acc=0.297
13:39:15 INFO [train_distill] [KD] epoch 6/50  loss=1.1875  val_acc=0.341
13:39:36 INFO [train_distill] [KD] epoch 7/50  loss=1.1721  val_acc=0.379
13:39:58 INFO [train_distill] [KD] epoch 8/50  loss=1.1537  val_acc=0.410
13:40:19 INFO [train_distill] [KD] epoch 9/50  loss=1.1364  val_ac

In [8]:
!cd soundedge-esc-application && \
    python -m compression.optimize --paper-mode \
        --csv ../data/fsc22/5-fold.csv \
        --audio-dir ../../DATA/FSC22/Audio/ \
        --weights weights/fsc22_model.pth \
        --out-dir  weights/paper-optimized \
        --val-fold 5 \
        --prune-step 0.1 \
        --prune-rounds 8 \
        --rewind-epochs 2 \
        --kd-epochs 30 \
        --qat-epochs 15 \
        --batch-size 32 --num-workers 8

14:00:06 INFO [main] device=cuda  smoke=False
14:00:06 INFO [main] num_classes=26  test_fold=None
14:00:06 WARNING [main] teacher weights not loaded (smoke or missing): weights/fsc22_model.pth
14:00:08 INFO [main] TEACHER  params=511805  val_acc=0.038
14:00:08 INFO [main] PAPER-MODE: iterative asymmetric prune (step=0.10, rounds<=8, target=off, rewind=2ep)
14:00:08 INFO [main]   round 1: params=468044
14:00:37 INFO [train_distill] [rewind1] epoch 1/2  loss=1.3544  val_acc=0.082
14:00:58 INFO [train_distill] [rewind1] epoch 2/2  loss=1.2717  val_acc=0.115
14:00:58 INFO [train_distill] [rewind1] best val_acc=0.115
14:00:58 INFO [main]   round 2: params=428462
14:01:20 INFO [train_distill] [rewind2] epoch 1/2  loss=1.2857  val_acc=0.141
14:01:42 INFO [train_distill] [rewind2] epoch 2/2  loss=1.2528  val_acc=0.149
14:01:42 INFO [train_distill] [rewind2] best val_acc=0.149
14:01:42 INFO [main]   round 3: params=396467
14:02:05 INFO [train_distill] [rewind3] epoch 1/2  loss=1.2526  val_acc=0

In [17]:
!cd soundedge-esc-application && \
    python -m compression.optimize --paper-mode --wandb \
        --wandb-project fsc22-optimize \
        --wandb-run paper-fold5 \
        --csv ../data/fsc22/5-fold.csv \
        --audio-dir ../../DATA/FSC22/Audio/ \
        --weights weights/fsc22_model.pth \
        --out-dir optimized-paper-50-target-50k \
        --val-fold 5 \
        --prune-step 0.1 \
        --target-params 52000 \
        --prune-rounds 22 \
        --rewind-epochs 2 \
        --kd-epochs 50 \
        --qat-epochs 15 \
        --batch-size 32 --num-workers 8

14:40:02 INFO [main] device=cuda  smoke=False
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/user/.netrc.
wandb: Currently logged in as: matogrod (matogrod-amu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run k94v3j2s (0.5s)
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in forest-sounds/soundedge-esc-application/wandb/run-20260601_144003-k94v3j2s
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run paper-fold5
wandb: ⭐️ View project at https://wandb.ai/matogrod-amu/fsc22-optimize
wandb: 🚀 View run at https://wandb.ai/matogrod-amu/fsc22-optimize/runs/k94v3j2s
14:40:05 INFO [main] num_classes=26  test_fold=None
14:40:05 WARNING [main] teacher weights not loaded (smoke or missing): weights/fsc22_model.pth
14:40:06 INFO [m

In [15]:
0.9**21

0.10941898913151242

In [ ]:
!cd soundedge-esc-application && \
    python -m compression.cv_optimize --paper-mode --wandb \
        --wandb-project fsc22-optimize \
        --wandb-run paper-fold5 \
        --csv ../data/fsc22/5-fold.csv \
        --audio-dir ../../DATA/FSC22/Audio/ \
        --weights weights/fsc22_model.pth \
        --out-dir optimized-paper-50-target-50k \
        --val-fold 5 \
        --prune-step 0.1 \
        --target-params 52000 \
        --prune-rounds 22 \
        --rewind-epochs 2 \
        --kd-epochs 50 \
        --qat-epochs 15 \
        --batch-size 32 --num-workers 8